In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

csv_path = Path(r"C:\Users\lfbou\Documents\GitHub\meteo-australie-prediction\notebooks\data\weatherAUS_encoded.csv")

df = pd.read_csv(csv_path, low_memory=False)
print(df.shape)
df.head(3)


In [ ]:

# 1) S'assurer que Year/Month sont numériques
df['Year']  = pd.to_numeric(df['Year'], errors='coerce').astype('Int64')
df['Month'] = pd.to_numeric(df['Month'], errors='coerce').astype('Int64')

# 2) Reconstituer une date (jour fixé à 1 : on veut l'ordre, pas le jour exact)
df['Date'] = pd.to_datetime(
    dict(year=df['Year'].astype('float').astype('Int64'),
         month=df['Month'].astype('float').astype('Int64'),
         day=1),
    errors='coerce'
)

# 3) Contrôles rapides et tri
assert df['Date'].notna().all(), "Certaines lignes ont Year/Month invalides → corriger avant d'avancer."
df = df.sort_values('Date').reset_index(drop=True)

print("Plage temporelle :", df['Date'].min(), "→", df['Date'].max())
print("Tri croissant par Date :", df['Date'].is_monotonic_increasing)


In [ ]:
df.head(3)


In [ ]:
import pandas as pd
from imblearn.under_sampling import RandomUnderSampler

# --- Paramètres
target_col = 'RainTomorrow'
test_months = 12  

# 1) Split temporel
df = df.sort_values('Date').reset_index(drop=True)
cutoff = df['Date'].max() - pd.DateOffset(months=test_months)
train_df = df[df['Date'] <= cutoff].copy()
test_df  = df[df['Date'] >  cutoff].copy()

feature_cols = [c for c in df.columns if c not in [target_col, 'Date']]
X_train, y_train = train_df[feature_cols], train_df[target_col].astype(int)
X_test,  y_test  = test_df[feature_cols],  test_df[target_col].astype(int)

print(f"Cutoff test (début): {cutoff.date()}")
print(f"Train: {train_df['Date'].min().date()} → {train_df['Date'].max().date()}  |  Test: {test_df['Date'].min().date()} → {test_df['Date'].max().date()}")
print(f"Taux positifs (train AVANT): {y_train.mean():.3f}  |  (test): {y_test.mean():.3f}")

#2) Rééquilibrage sur le TRAIN uniquement (under-sampling léger)
rus = RandomUnderSampler(sampling_strategy=0.8, random_state=42)
X_train_bal, y_train_bal = rus.fit_resample(X_train, y_train)

# remettre en DataFrame pour garder les noms de colonnes
X_train_bal = pd.DataFrame(X_train_bal, columns=feature_cols)
y_train_bal = pd.Series(y_train_bal, name=target_col)

print(f"Tailles: X_train={X_train.shape},  X_train_bal={X_train_bal.shape},  X_test={X_test.shape}")
print(f"Taux positifs (train APRÈS): {y_train_bal.mean():.3f}")


In [ ]:
# on peut maintenant entraîner un modèle sur (X_train_bal, y_train_bal) et évaluer sur (X_test, y_test)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

logreg = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=300, class_weight="balanced", solver="lbfgs"))
])

logreg.fit(X_train_bal, y_train_bal)
y_pred = logreg.predict(X_test)

print("=== Régression logistique ===")
print(classification_report(y_test, y_pred, target_names=["pas de pluie","pluie"]))
print("F1 (classe pluie=1):", f1_score(y_test, y_pred, pos_label=1))


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score

rf = RandomForestClassifier(
    n_estimators=400,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train_bal, y_train_bal)
y_pred = rf.predict(X_test)

print("=== Forêt aléatoire ===")
print(classification_report(y_test, y_pred, target_names=["pas de pluie","pluie"]))
print("F1 (classe pluie=1):", f1_score(y_test, y_pred, pos_label=1))


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, f1_score

gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train_bal, y_train_bal)
y_pred = gb.predict(X_test)

print("=== Gradient Boosting ===")
print(classification_report(y_test, y_pred, target_names=["pas de pluie","pluie"]))
print("F1 (classe pluie=1):", f1_score(y_test, y_pred, pos_label=1))


In [ ]:

from sklearn.metrics import (
    classification_report, f1_score, precision_recall_fscore_support,
    roc_auc_score, average_precision_score
)

# 1) Validation temporelle à l’intérieur du train (2 derniers mois)
last_train_date = train_df["Date"].max()
val_cut = last_train_date - pd.DateOffset(months=2)

X_tr  = train_df[train_df["Date"] <= val_cut][feature_cols]
y_tr  = train_df[train_df["Date"] <= val_cut]["RainTomorrow"].astype(int)
X_val = train_df[train_df["Date"] >  val_cut][feature_cols]
y_val = train_df[train_df["Date"] >  val_cut]["RainTomorrow"].astype(int)

# 2) Rééquilibrage uniquement sur la partie apprentissage de ce split
rus = RandomUnderSampler(sampling_strategy=0.9, random_state=42)
X_tr_bal, y_tr_bal = rus.fit_resample(X_tr, y_tr)

# 3) Entraîner la forêt sur (X_tr_bal, y_tr_bal)
rf = RandomForestClassifier(
    n_estimators=600,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=42
)
rf.fit(X_tr_bal, y_tr_bal)

# 4) Choisir le seuil qui maximise le F1 (classe pluie=1) sur la validation
proba_val = rf.predict_proba(X_val)[:, 1]
best_t, best_f1 = 0.5, -1
for t in np.linspace(0.2, 0.8, 61):
    y_hat = (proba_val >= t).astype(int)
    _, _, f1, _ = precision_recall_fscore_support(y_val, y_hat, average='binary', pos_label=1)
    if f1 > best_f1:
        best_t, best_f1 = t, f1

print(f"[VALIDATION] Seuil choisi = {best_t:.2f} | F1(pluie) = {best_f1:.3f}")

# 5) Ré-entraîner sur TOUT le train rééquilibré global (X_train_bal, y_train_bal), puis évaluer sur le test gelé
rf.fit(X_train_bal, y_train_bal)

proba_test = rf.predict_proba(X_test)[:, 1]
y_pred_test = (proba_test >= best_t).astype(int)

print("\n[TEST] Rapport de classification (seuil optimisé)")
print(classification_report(y_test, y_pred_test, target_names=["pas de pluie","pluie"]))
print("F1 (classe pluie=1):", f1_score(y_test, y_pred_test, pos_label=1))
print("ROC-AUC:", roc_auc_score(y_test, proba_test))
print("PR-AUC (Average Precision):", average_precision_score(y_test, proba_test))

# 6) Top 10 importances de variables (pour interpréter)
imp = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False).head(10)
print("\nTop 10 importances de variables :")
print(imp)


In [ ]:
import numpy as np
import pandas as pd
from imblearn.under_sampling import RandomUnderSampler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    precision_recall_fscore_support, classification_report,
    roc_auc_score, average_precision_score, f1_score
)

# 1) Validation temporelle = 2 derniers mois du train
last_train_date = train_df["Date"].max()
val_cut = last_train_date - pd.DateOffset(months=2)

X_tr  = train_df[train_df["Date"] <= val_cut][feature_cols]
y_tr  = train_df[train_df["Date"] <= val_cut]["RainTomorrow"].astype(int)
X_val = train_df[train_df["Date"] >  val_cut][feature_cols]
y_val = train_df[train_df["Date"] >  val_cut]["RainTomorrow"].astype(int)

# Rééquilibrage uniquement sur la partie apprentissage de CE split
rus = RandomUnderSampler(sampling_strategy=0.9, random_state=42)
X_tr_bal, y_tr_bal = rus.fit_resample(X_tr, y_tr)

# 2) Petite grille d’hyperparamètres "sobre"
grid = {
    "learning_rate": [0.05, 0.1],
    "n_estimators": [300, 500],
    "max_depth": [2, 3],
    "min_samples_leaf": [1, 5],
    "subsample": [0.8, 1.0],
    "max_features": [None, "sqrt"],
}
def param_product(grid):
    import itertools
    keys = list(grid.keys())
    for values in itertools.product(*[grid[k] for k in keys]):
        yield dict(zip(keys, values))

best = {"f1_val": -1, "thr": 0.5, "params": None, "model": None}

for params in param_product(grid):
    gb = GradientBoostingClassifier(random_state=42, **params)
    gb.fit(X_tr_bal, y_tr_bal)

    # 3) Choix du seuil sur la validation (max F1 classe "pluie" = 1)
    proba_val = gb.predict_proba(X_val)[:, 1]
    best_t, best_f1 = 0.5, -1
    for t in np.linspace(0.2, 0.8, 61):
        y_hat = (proba_val >= t).astype(int)
        _, _, f1, _ = precision_recall_fscore_support(y_val, y_hat, average='binary', pos_label=1)
        if f1 > best_f1:
            best_t, best_f1 = t, f1

    if best_f1 > best["f1_val"]:
        best = {"f1_val": best_f1, "thr": best_t, "params": params, "model": gb}

print(f"[VALIDATION] Meilleur F1(pluie)={best['f1_val']:.3f} @ seuil={best['thr']:.2f}")
print("[VALIDATION] Paramètres retenus:", best["params"])

# 4) Ré-entraîner sur TOUT le train rééquilibré global puis évaluer sur le test gelé
gb_final = GradientBoostingClassifier(random_state=42, **best["params"])
gb_final.fit(X_train_bal, y_train_bal)

proba_test = gb_final.predict_proba(X_test)[:, 1]
y_pred_test = (proba_test >= best["thr"]).astype(int)

print("\n[TEST] Rapport de classification (seuil optimisé)")
print(classification_report(y_test, y_pred_test, target_names=["pas de pluie","pluie"]))
print("F1 (classe pluie=1):", f1_score(y_test, y_pred_test, pos_label=1))
print("ROC-AUC:", roc_auc_score(y_test, proba_test))
print("PR-AUC (Average Precision):", average_precision_score(y_test, proba_test))

# 5) Top 10 importances (pour interprétation dans le rapport)
imp = pd.Series(gb_final.feature_importances_, index=feature_cols).sort_values(ascending=False).head(10)
print("\nTop 10 importances de variables :")
print(imp)


In [ ]:
import numpy as np
import pandas as pd
from imblearn.under_sampling import RandomUnderSampler
from sklearn.experimental import enable_hist_gradient_boosting  # noqa
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    precision_recall_fscore_support, classification_report,
    roc_auc_score, average_precision_score, f1_score
)


df_work = df.copy()

# -------------------------
# 1) Reconstituer "Location" à partir des one-hot Location_*
# -------------------------
loc_cols = [c for c in df_work.columns if c.startswith("Location_")]
if len(loc_cols) == 0:
    raise ValueError("Aucune colonne Location_* trouvée. Ajoute-les ou fournis la colonne Location d'origine.")

# argmax sur les colonnes one-hot pour retrouver l'étiquette
loc_idx = np.argmax(df_work[loc_cols].values, axis=1)
labels = [c.replace("Location_", "") for c in loc_cols]
df_work["Location"] = [labels[i] for i in loc_idx]

# -------------------------
# 2) Créer des features de mémoire PAR STATION, triées par Date
# -------------------------
base_feats = ["Humidity3pm","Humidity9am","Pressure3pm","Rainfall","WindGustSpeed","Temp3pm"]
missing = [c for c in base_feats if c not in df_work.columns]
if missing:
    raise ValueError(f"Colonnes manquantes pour features mémoire: {missing}")

df_work = df_work.sort_values(["Location","Date"]).reset_index(drop=True)

# lags J-1 / J-2
for col in base_feats:
    df_work[f"{col}_lag1"] = df_work.groupby("Location")[col].shift(1)
    df_work[f"{col}_lag2"] = df_work.groupby("Location")[col].shift(2)

# différences jour-à-jour
for col in base_feats:
    df_work[f"{col}_diff1"] = df_work[col] - df_work[f"{col}_lag1"]

# roulantes (fenêtre 3 jours, passé uniquement)
roll_window = 3
for col in ["Humidity3pm","Pressure3pm","Rainfall"]:
    df_work[f"{col}_roll{roll_window}"] = (
        df_work.groupby("Location")[col]
        .apply(lambda s: s.shift(1).rolling(roll_window, min_periods=roll_window).mean())
        .reset_index(level=0, drop=True)
    )

# Compte de jours pluvieux sur 3 jours (Rainfall >= 1mm)
df_work["RainDay"] = (df_work["Rainfall"] >= 1.0).astype(int)
df_work["RainDays_roll3"] = (
    df_work.groupby("Location")["RainDay"]
    .apply(lambda s: s.shift(1).rolling(roll_window, min_periods=roll_window).sum())
    .reset_index(level=0, drop=True)
)

# Drop premières lignes de chaque station sans historique suffisant
df_work = df_work.groupby("Location", group_keys=False).apply(lambda g: g.iloc[2:]).reset_index(drop=True)

# -------------------------
# 3) Split temporel: 12 derniers mois = test
# -------------------------
df_work = df_work.sort_values("Date").reset_index(drop=True)
target_col = "RainTomorrow"

cutoff = df_work["Date"].max() - pd.DateOffset(months=12)
train_df = df_work[df_work["Date"] <= cutoff].copy()
test_df  = df_work[df_work["Date"] >  cutoff].copy()

feature_cols = [c for c in df_work.columns if c not in [target_col, "Date", "Location"]]

X_train = train_df[feature_cols]
y_train = train_df[target_col].astype(int)
X_test  = test_df[feature_cols]
y_test  = test_df[target_col].astype(int)

print(f"Cutoff test: {cutoff.date()} | Train: {train_df['Date'].min().date()}→{train_df['Date'].max().date()} | Test: {test_df['Date'].min().date()}→{test_df['Date'].max().date()}")
print(f"Taux positifs (train): {y_train.mean():.3f} | (test): {y_test.mean():.3f}")

# -------------------------
# 4) Validation temporelle dans le TRAIN + undersampling uniquement côté apprentissage
# -------------------------
last_train_date = train_df["Date"].max()
val_cut = last_train_date - pd.DateOffset(months=2)

X_tr  = train_df[train_df["Date"] <= val_cut][feature_cols]
y_tr  = train_df[train_df["Date"] <= val_cut][target_col].astype(int)
X_val = train_df[train_df["Date"] >  val_cut][feature_cols]
y_val = train_df[train_df["Date"] >  val_cut][target_col].astype(int)

rus = RandomUnderSampler(sampling_strategy=0.9, random_state=42)
X_tr_bal, y_tr_bal = rus.fit_resample(X_tr, y_tr)

print(f"Taux positifs apprentissage (avant): {y_tr.mean():.3f} | (après): {y_tr_bal.mean():.3f} | Validation (naturelle): {y_val.mean():.3f}")

# -------------------------
# 5) HistGradientBoostingClassifier + petite grille + seuil optimisé sur VALIDATION
# -------------------------
grids = [
    {"learning_rate": 0.05, "max_iter": 600, "max_leaf_nodes": 31, "min_samples_leaf": 20, "max_depth": None, "l2_regularization": 0.0},
    {"learning_rate": 0.10, "max_iter": 500, "max_leaf_nodes": 31, "min_samples_leaf": 20, "max_depth": None, "l2_regularization": 0.1},
    {"learning_rate": 0.05, "max_iter": 600, "max_leaf_nodes": 15, "min_samples_leaf": 50, "max_depth": 6,    "l2_regularization": 0.0},
]

best = {"f1_val": -1, "thr": 0.5, "params": None}

for params in grids:
    hgb = HistGradientBoostingClassifier(early_stopping=False, random_state=42, **params)
    hgb.fit(X_tr_bal, y_tr_bal)

    proba_val = hgb.predict_proba(X_val)[:, 1]
    # balayage fin du seuil autour de 0.5 (élargissable)
    best_t, best_f1 = 0.5, -1
    for t in np.linspace(0.4, 0.7, 31):
        y_hat = (proba_val >= t).astype(int)
        _, _, f1, _ = precision_recall_fscore_support(y_val, y_hat, average='binary', pos_label=1)
        if f1 > best_f1:
            best_t, best_f1 = t, f1

    if best_f1 > best["f1_val"]:
        best = {"f1_val": best_f1, "thr": best_t, "params": params}

print(f"[VALIDATION] Meilleur F1(pluie)={best['f1_val']:.3f} @ seuil={best['thr']:.2f}")
print("[VALIDATION] Paramètres retenus:", best["params"])

# -------------------------
# 6) Ré-entraîner sur TOUT le train (undersampling global léger) + évaluation sur TEST gelé
# -------------------------
rus_global = RandomUnderSampler(sampling_strategy=0.9, random_state=42)
X_train_bal, y_train_bal = rus_global.fit_resample(X_train, y_train)

hgb_final = HistGradientBoostingClassifier(early_stopping=False, random_state=42, **best["params"])
hgb_final.fit(X_train_bal, y_train_bal)

proba_test = hgb_final.predict_proba(X_test)[:, 1]
y_pred_test = (proba_test >= best["thr"]).astype(int)

print("\n[TEST] Rapport de classification (seuil optimisé)")
print(classification_report(y_test, y_pred_test, target_names=["pas de pluie","pluie"]))
print("F1 (classe pluie=1):", f1_score(y_test, y_pred_test, pos_label=1))
print("ROC-AUC:", roc_auc_score(y_test, proba_test))
print("PR-AUC (Average Precision):", average_precision_score(y_test, proba_test))
